In [0]:

CREATE OR REPLACE VIEW
pondolens.dev_gold.vw_dpwh_regional_spending
AS

WITH dpwh_obligations AS (
    SELECT
        dy.year,
        dl.location_id,
        dl.location_code,
        dl.location,

        SUM(fe.obligation_amount)
            AS dpwh_obligations

    FROM pondolens.dev_gold.fact_regional_expenditure fe

    INNER JOIN pondolens.dev_gold.dim_particular dp
        ON fe.particular_id = dp.particular_id

    INNER JOIN pondolens.dev_gold.dim_location dl
        ON fe.location_id = dl.location_id

    INNER JOIN pondolens.dev_gold.dim_year dy
        ON fe.year_id = dy.year_id

    WHERE
        UPPER(TRIM(dp.category)) = 'DEPARTMENTS'
        AND UPPER(TRIM(dp.particular))
            LIKE 'DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS%'

        AND UPPER(TRIM(dl.location_code)) <> 'PHL'
        AND dy.year BETWEEN 2021 AND 2025

    GROUP BY
        dy.year,
        dl.location_id,
        dl.location_code,
        dl.location
),

annual_totals AS (
    SELECT
        year,
        SUM(dpwh_obligations) AS annual_dpwh_obligations
    FROM dpwh_obligations
    GROUP BY year
)

SELECT
    d.year,
    d.location_id,
    d.location_code,
    d.location,
    d.dpwh_obligations,
    t.annual_dpwh_obligations,

    ROUND(
        d.dpwh_obligations * 100.0
        / NULLIF(t.annual_dpwh_obligations, 0),
        2
    ) AS spending_share_pct,

    DENSE_RANK() OVER (
        PARTITION BY d.year
        ORDER BY d.dpwh_obligations DESC
    ) AS spending_rank

FROM dpwh_obligations d

INNER JOIN annual_totals t
    ON d.year = t.year;
